<a href="https://colab.research.google.com/github/lcbjrrr/genai/blob/main/MCPGemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP

## MCP Gemini



The **Model Context Protocol (MCP)** is an open-source standard developed by Anthropic to solve the "data silo" problem in the AI ecosystem by providing a universal way for Large Language Models (LLMs) to connect with external data sources and tools. Traditionally, developers had to write custom, one-off integrations for every unique data source (like Google Drive, GitHub, or Slack) for each specific AI application. MCP replaces this fragmented approach with a standardized client-server architecture where a single "MCP Server" can expose data and functionality to any "MCP Client" (such as Claude Desktop or an IDE). This allows AI models to securely and dynamically retrieve context, execute code, and interact with local or remote services through a consistent, plug-and-play interface, significantly reducing the complexity of building sophisticated AI agents.

In the MCP architecture, a remote configuration, specifically a Large Language Model (LLM) interacts with external tools across the web. The AI model doesn't need to have the tools built into its own code. Instead, it reaches out over the internet to a specialized server that manages those tools for it.



![](https://pbs.twimg.com/media/HGSI9DaXwAEfdHt?format=jpg&name=small)

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("simple-mcp")

@mcp.tool()
def state_capital_f(state: str) -> str:
    """Provides the most up-to-date capital for a given US state."""
    state_capital_dict = {
        "Michigan": "Lansing",
        "Florida": "Tallahassee",
        "California": "Sacramento",
        "Texas": "Austin",
        "New York": "Albany"
    }
    capital = state_capital_dict.get(state, "Unknown")
    print(f"------> {state}: {capital}")
    return capital

@mcp.tool()
def temperature_f(city: str) -> int:
    """Provides the historical average temperature for a given city."""
    temp_dict = {
        "Lansing": 50,
        "Tallahassee": 90,
        "Sacramento": 85,
        "Austin": 80,
        "Albany": 45
    }
    temp = temp_dict.get(city, "Data not available")
    print(f"=====> {city}: {temp}")
    return temp

if __name__ == "__main__":
    mcp.run(transport="sse")

It is offered in three transport protocols

| Feature | **Stdio** | **HTTP + SSE** | **Http-Steam** |
| :--- | :--- | :--- | :--- |
| **Primary Use Case** | Local development and CLI tools (e.g., Claude Desktop). | Remote/Cloud tools and web-compatible integrations. | Real-time, high-frequency bidirectional apps. |
| **Environment** | Same machine (Local). | Distributed (Network/Internet). | Distributed (Network/Internet). |
| **Communication** | Bidirectional via `stdin`/`stdout`. | Unidirectional stream (Server → Client); POST for Client → Server. | Full-duplex (simultaneous bidirectional). |
| **Lifecycle** | Host starts/stops the process. | Server runs as an independent web service. | Persistent, stateful independent connection. |
| **Ease of Setup** | **High** (No network/ports). | **Medium** (Requires web server logic). | **Low** (Requires handshake and state mgmt). |
| **Scalability** | Single client only. | Many clients; load-balancer friendly. | Many clients; requires "sticky" sessions. |

# Local MCP Server

Some LLM providers does not allow to connect to MCP servers directly on the Web UI. For security reasons, it only allows to connect those through its local clients (or similar products). An LLM client (like the Gemini CLI or Claude Code) acts as the central orchestrator that manages the conversation by hosting the AI model, discovering available tools from MCP servers, and translating the model's intent into specific actions, like querying your databases or reading local files, to provide a context-aware response.

![](https://pbs.twimg.com/media/HGSMsMPXEAAg6M8?format=jpg&name=small)

To register your MCP to Gemini CLI `~/.gemini/settings.json`.



```
  "mcpServers": {
    "simple-mcp": {
      "url": "http://localhost:8000/sse",
      "type": "sse"
    }
  }
```


> What is the Michigan capital? And what is the average temperature in there?





```

 ▝▜▄    Gemini CLI v0.38.2
   ▝▜▄
   ▗▟▀  Signed in with Google /auth
 ▗▟▀    Plan: Gemini Code Assist in Google One AI Pro /upgrade

▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
 > /mcp                                                                                                               
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
Configured MCP servers:

🟢 simple-mcp - Ready (2 tools)
  Tools:
  - mcp_simple-mcp_state_capital_f
  - mcp_simple-mcp_temperature_f


```





## Remote MCP Server

When an MCP server is deployed with a public URL, typically using the HTTP with SSE transport, it becomes a versatile resource that can be accessed by both cloud-based and local environments. In this configuration, LLM Web UIs (from providers that support remote MCP integration) can reach out across the internet to consume the server's tools and resources directly within their hosted chat interfaces. Simultaneously, a local client, such as an IDE or a customized Gemini CLI running on your own machine, can connect to that same public endpoint, allowing you to maintain a unified toolset across different development contexts without needing to host multiple local instances of the server.

![](https://pbs.twimg.com/media/HGSSDNkWkAAGJ-b?format=png&name=900x900)



> http://127.0.0.1:8000/sse



```
#########

curl -X POST "http://127.0.0.1:8000/messages/?session_id=38c8f02396384cb580fcb098ab8d68eb" \
  -H "Content-Type: application/json" \
  -d '{
    "jsonrpc":"2.0",
    "id":1,
    "method":"initialize",
    "params":{
      "protocolVersion":"2025-06-18",
      "capabilities":{"tools":{}},
      "clientInfo":{"name":"curl","version":"1"}
    }
  }'

curl -X POST "http://127.0.0.1:8000/messages/?session_id=38c8f02396384cb580fcb098ab8d68eb" \
  -H "Content-Type: application/json" \
  -d '{
    "jsonrpc":"2.0",
    "method":"notifications/initialized",
    "params":{}
  }'

curl -X POST "http://127.0.0.1:8000/messages/?session_id=38c8f02396384cb580fcb098ab8d68eb" \
  -H "Content-Type: application/json" \
  -d '{
    "jsonrpc":"2.0",
    "id":2,
    "method":"tools/list",
    "params":{}
  }'

```



## AI Agent calling MCP Tools

While AI agents are often built with a set of internal, hard-coded tools, the Model Context Protocol (MCP) provides a standardized bridge that allows these agents to look beyond their local capabilities. Because MCP is an open standard, any agentic system—whether it’s a simple script or a complex, can act as an MCP client. This means that instead of developers writing custom integrations for every new agent, the agents can simply "plug in" to a remote MCP server to instantly inherit access to a wide array of databases, APIs, and file systems. This interoperability ensures that specialized tools, like the MySQL or PostgreSQL utilities you've developed, can be shared across multiple different agentic architectures through a single, consistent interface.

![](https://pbs.twimg.com/media/HGSRt9lXUAA7L1A?format=jpg&name=medium)

In [ ]:
#pip install langchain_mcp_adapters langchain_google_genai langchain


import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent  # ✅ updated import
import os
API_KEY='-cI'
os.environ["GOOGLE_API_KEY"] = API_KEY

async def main():
    client = MultiServerMCPClient({"simple-mcp": {"url": "http://localhost:8000/sse","transport": "sse",}})
    tools = await client.get_tools()
    gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    agent = create_agent(model=gemini,tools=tools,system_prompt="You are a helpful assistant.")
    prompt="What is the Michigan capital? And what is the average temperature in there?"
    response = await agent.ainvoke({"messages": [{"role": "user", "content": prompt}]})
    print(response["messages"][-1].content[0]['text'])
asyncio.run(main())

# Yet another example

### MCP Server

In [ ]:
from mcp.server.fastmcp import FastMCP

import sqlite3
from langchain.tools import tool
import pandas as pd

TABLES = {
    'teams': "Stores the teams' names and their nicknames",
    'players': "Contains the main players along with their numbers for every team",
    'coaches': "Holds the record of the head coach for each team"
}

def get_table_context(db, table_name, table_description):
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    print(f"PRAGMA table_info({table_name});")
    cursor.execute(f"PRAGMA table_info({table_name});")
    schema = cursor.fetchall()

    columns_str = ""
    for col in schema:
        columns_str += f" **{col[1]}**: {col[2]};"

    context = f"""Here is the table name ***{table_name}***: *{table_description}*.
    Here are the columns of the {table_name}: {columns_str}"""
    print('---->', context)
    return context

mcp = FastMCP("sports-mcp")

@mcp.tool()
def inspect_db_tool(x="") :
    """Inspect the database and retrieve each table's name, description, and column schema."""
    db = 'teams.db'
    table_context = ""
    for name, desc in TABLES.items():
        table_context += '\n\n' + get_table_context(db, name, desc)
    return table_context

@mcp.tool()
def execute_query_tool(query):
    """Execute a SQL query in the local teams.db database and return the result as a pandas DataFrame."""
    DBNAME = 'teams.db'
    print("======>", query)
    conn = sqlite3.connect(DBNAME)
    cursor = conn.cursor()
    cursor.execute(query)
    return pd.DataFrame(cursor.fetchall())

if __name__ == "__main__":
    mcp.run(transport="sse")

#Find on the database the mascots of these teams Navy and Troy. Considering that the score was 22 x 11, create a message to be posted on social media mentioning the teams mascots. Return only the post no intros or questions")

### Agent Client (optional)

In [ ]:
#pip install langchain_mcp_adapters langchain_google_genai langchain



import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent  # ✅ updated import
import os
API_KEY='xxxx-cI'
os.environ["GOOGLE_API_KEY"] = API_KEY

async def main():
    client = MultiServerMCPClient(
        {
            "simple-mcp": {
                "url": "http://localhost:8000/sse",
                "transport": "sse",
            }
        }
    )

    tools = await client.get_tools()

    gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

    agent = create_agent(  # ✅ updated function name
        model=gemini,
        tools=tools,
        system_prompt="You are a helpful assistant."
    )

    prompt="Find on the database the mascots of these teams Navy and Troy. Considering that the score was 22 x 11, create a message to be posted on social media mentioning the teams mascots. Return only the post no intros or questions"
    response = await agent.ainvoke({
        "messages": [{"role": "user", "content": prompt}]
    })

    print(response["messages"][-1].content[0]['text'])

asyncio.run(main())

## APPENDIX



```

curl -i -X POST http://localhost:8000/mcp \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{
    "jsonrpc":"2.0",
    "id":1,
    "method":"initialize",
    "params":{
      "protocolVersion":"2025-06-18",
      "capabilities":{"tools":{}},
      "clientInfo":{"name":"curl","version":"1"}
    }
  }'


SESSION_ID="254ce8c95fca446cbd3d52f6d5c4cf96"
curl -i -X POST "http://localhost:8000/mcp" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -H "Mcp-Session-Id: $SESSION_ID" \
  -d '{
    "jsonrpc":"2.0",
    "method":"notifications/initialized",
    "params":{}
  }'

SESSION_ID="254ce8c95fca446cbd3d52f6d5c4cf96"
curl -Ns --max-time 30 -X POST "http://localhost:8000/mcp" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -H "Mcp-Session-Id: $SESSION_ID" \
  -d '{"jsonrpc":"2.0","id":2,"method":"tools/list","params":{}}' | head -50

curl -i -X POST "http://localhost:8000/mcp" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -H "Mcp-Session-Id: $SESSION_ID" \
  -d '{
    "jsonrpc":"2.0",
    "id":3,
    "method":"tools/call",
    "params":{
      "name":"get_weather",
      "arguments":{"city":"Boston"}
    }
  }'

```



In [ ]:

# curl -i -X POST http://localhost:8000/mcp \
#   -H "Content-Type: application/json" \
#   -H "Accept: application/json, text/event-stream" \
#   -d '{
#     "jsonrpc":"2.0",
#     "id":1,
#     "method":"initialize",
#     "params":{
#       "protocolVersion":"2025-06-18",
#       "capabilities":{"tools":{}},
#       "clientInfo":{"name":"curl","version":"1"}
#     }
#   }'


# SESSION_ID="254ce8c95fca446cbd3d52f6d5c4cf96"
# curl -i -X POST "http://localhost:8000/mcp" \
#   -H "Content-Type: application/json" \
#   -H "Accept: application/json, text/event-stream" \
#   -H "Mcp-Session-Id: $SESSION_ID" \
#   -d '{
#     "jsonrpc":"2.0",
#     "method":"notifications/initialized",
#     "params":{}
#   }'

# SESSION_ID="254ce8c95fca446cbd3d52f6d5c4cf96"
# curl -Ns --max-time 30 -X POST "http://localhost:8000/mcp" \
#   -H "Content-Type: application/json" \
#   -H "Accept: application/json, text/event-stream" \
#   -H "Mcp-Session-Id: $SESSION_ID" \
#   -d '{"jsonrpc":"2.0","id":2,"method":"tools/list","params":{}}' | head -50

# curl -i -X POST "http://localhost:8000/mcp" \
#   -H "Content-Type: application/json" \
#   -H "Accept: application/json, text/event-stream" \
#   -H "Mcp-Session-Id: $SESSION_ID" \
#   -d '{
#     "jsonrpc":"2.0",
#     "id":3,
#     "method":"tools/call",
#     "params":{
#       "name":"get_weather",
#       "arguments":{"city":"Boston"}
#     }
#   }'





```
#python -m pip install "mcp[cli]"



from mcp.server.fastmcp import FastMCP

mcp = FastMCP("simple-mcp")

@mcp.tool()
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"Weather for {city}: sunny"

@mcp.tool()
def calc_total(price: float, tax: float) -> float:
    """Calculate the final total after tax."""
    return price + tax

if __name__ == "__main__":
    #mcp.run(transport="streamable-http")
    mcp.run(transport="sse")

#CLIENT
#pip install langchain_mcp_adapters langchain_google_genai langchain



import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent  # ✅ updated import
import os
API_KEY='-cI'
os.environ["GOOGLE_API_KEY"] = API_KEY

async def main():
    client = MultiServerMCPClient(
        {
            "simple-mcp": {
                "url": "http://localhost:8000/sse",
                "transport": "sse",
            }
        }
    )

    tools = await client.get_tools()

    gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

    agent = create_agent(  # ✅ updated function name
        model=gemini,
        tools=tools,
        system_prompt="You are a helpful assistant."
    )

    response = await agent.ainvoke({
        "messages": [{"role": "user", "content": "What is the weather in Paris?"}]
    })

    print(response["messages"][-1].content)

asyncio.run(main())
```

